# Feature Engineering (TF-IDF)

In [111]:
import pandas as pd
import numpy as np
import os
import json
import joblib
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

DATA_PATH = '../data/processed/labeled_comments_kopdes.parquet'
PREP_PATH = '../data/interim/preprocessed_comments_kopdes.parquet'

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f'File {DATA_PATH} belum ada.\n'
        'Jalankan dulu 06_sentiment_labeling_indobert.ipynb di Google Colab.'
    )

df_labeled = pd.read_parquet(DATA_PATH)
df_prep = pd.read_parquet(PREP_PATH)[['video_id', 'comment_id', 'text_final']]

df = df_labeled.merge(df_prep, on=['video_id', 'comment_id'], how='inner')
print(f'Baris untuk feature engineering: {len(df):,}')
assert set(df['sentiment_label'].unique()) <= {'positive', 'neutral', 'negative'}
df[['text_final', 'sentiment_label']].head(5)

Baris untuk feature engineering: 118,778


,text_final,sentiment_label
0,beliau bilang untung,positive
1,makasih mas,positive
2,untung barang jual ajh,negative
3,bilang makasih mas,neutral
4,niat bantu umkm harus koprasi merah putih buka bentuk warehouse gudang bukan bentuk toko jadi be...,positive


## Train-Test Split (Stratified)


In [112]:
X_text = df['text_final']
y = df['sentiment_label']

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {len(X_train_text):,} baris | Test: {len(X_test_text):,} baris')
print()
print('Distribusi kelas - Train:')
print(y_train.value_counts(normalize=True).round(3))
print()
print('Distribusi kelas - Test:')
print(y_test.value_counts(normalize=True).round(3))
print()
print('-> Proporsi kelas antara train & test seimbang (hasil stratified split).')

Train: 95,022 baris | Test: 23,756 baris

Distribusi kelas - Train:
sentiment_label
negative    0.582
neutral     0.233
positive    0.185
Name: proportion, dtype: float64

Distribusi kelas - Test:
sentiment_label
negative    0.582
neutral     0.233
positive    0.185
Name: proportion, dtype: float64

-> Proporsi kelas antara train & test seimbang (hasil stratified split).


## Fit TF-IDF Vectorizer (pada Train Saja)


In [113]:
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.9,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

print(f'Ukuran vocabulary TF-IDF : {len(tfidf.vocabulary_):,}')
print(f'Shape X_train            : {X_train_tfidf.shape}')
print(f'Shape X_test             : {X_test_tfidf.shape}')
print(f'Sparsity X_train         : {1 - X_train_tfidf.nnz / (X_train_tfidf.shape[0]*X_train_tfidf.shape[1]):.4f}')

Ukuran vocabulary TF-IDF : 10,000
Shape X_train            : (95022, 10000)
Shape X_test             : (23756, 10000)
Sparsity X_train         : 0.9993


## Validasi Kualitas Fitur


In [114]:
feature_names = np.array(tfidf.get_feature_names_out())
mean_tfidf = np.asarray(X_train_tfidf.mean(axis=0)).ravel()
top_idx = mean_tfidf.argsort()[::-1][:20]

print('Top 20 term dengan rata-rata skor TF-IDF tertinggi (data train):')
for i in top_idx:
    print(f'  {feature_names[i]:25s} {mean_tfidf[i]:.4f}')

assert X_train_tfidf.shape[0] == len(y_train)
assert X_test_tfidf.shape[0] == len(y_test)
assert X_train_tfidf.shape[1] == X_test_tfidf.shape[1]
print()
print('VALIDASI PASSED: dimensi fitur train/test konsisten, tidak ada mismatch baris.')

Top 20 term dengan rata-rata skor TF-IDF tertinggi (data train):
  apa                       0.0117
  mau                       0.0107
  warung                    0.0103
  harga                     0.0100
  kak                       0.0097
  tutup                     0.0094
  indomaret                 0.0094
  beli                      0.0092
  koperasi                  0.0081
  barang                    0.0080
  jual                      0.0078
  alfa                      0.0077
  kopdes                    0.0076
  saing                     0.0075
  alfamart                  0.0074
  pak                       0.0074
  toko                      0.0074
  rakyat                    0.0074
  jadi                      0.0073
  buka                      0.0073

VALIDASI PASSED: dimensi fitur train/test konsisten, tidak ada mismatch baris.


## Simpan Vectorizer & Fitur

In [115]:
os.makedirs('models', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

joblib.dump(tfidf, '../models/tfidf_vectorizer.pkl')

sp.save_npz('../data/processed/X_train_tfidf.npz', X_train_tfidf)
sp.save_npz('../data/processed/X_test_tfidf.npz', X_test_tfidf)
y_train.to_frame(name='sentiment_label').to_csv('../data/processed/y_train.csv', index=False)
y_test.to_frame(name='sentiment_label').to_csv('../data/processed/y_test.csv', index=False)

X_train_text.to_frame(name='text_final').assign(sentiment_label=y_train.values).to_csv(
    '../data/processed/train_text_reference.csv', index=False)
X_test_text.to_frame(name='text_final').assign(sentiment_label=y_test.values).to_csv(
    '../data/processed/test_text_reference.csv', index=False)

feature_engineering_summary = {
    'method': 'TF-IDF',
    'vocab_size': len(tfidf.vocabulary_),
    'ngram_range': [1, 2],
    'max_features': 10000,
    'min_df': 3,
    'max_df': 0.9,
    'n_train': int(X_train_tfidf.shape[0]),
    'n_test': int(X_test_tfidf.shape[0]),
    'train_class_dist': y_train.value_counts().to_dict(),
    'test_class_dist': y_test.value_counts().to_dict(),
}
with open('../reports/feature_engineering_summary.json', 'w') as f:
    json.dump(feature_engineering_summary, f, indent=2, default=str)

print(json.dumps(feature_engineering_summary, indent=2, default=str))
print()
print('Tersimpan: models/tfidf_vectorizer.pkl, data/processed/X_train_tfidf.npz, X_test_tfidf.npz')

{
  "method": "TF-IDF",
  "vocab_size": 10000,
  "ngram_range": [
    1,
    2
  ],
  "max_features": 10000,
  "min_df": 3,
  "max_df": 0.9,
  "n_train": 95022,
  "n_test": 23756,
  "train_class_dist": {
    "negative": 55266,
    "neutral": 22154,
    "positive": 17602
  },
  "test_class_dist": {
    "negative": 13817,
    "neutral": 5538,
    "positive": 4401
  }
}

Tersimpan: models/tfidf_vectorizer.pkl, data/processed/X_train_tfidf.npz, X_test_tfidf.npz


## Kesimpulan Tahap Feature Engineering

- TF-IDF di-*fit* hanya pada data train (tidak ada data leakage dari test set)
- Vocabulary size dan parameter tercatat untuk reproducibility
- Vectorizer disimpan (`models/tfidf_vectorizer.pkl`) agar bisa dipakai ulang saat Deployment
  Streamlit (Tahap 13) untuk mengubah input pengguna jadi vektor fitur yang sama
- File `train_text_reference.csv` / `test_text_reference.csv` disimpan untuk keperluan analisis
  kesalahan model nanti (menelusuri balik prediksi salah ke teks aslinya)

**Status: Tahap 8 (Feature Engineering) selesai.** Lanjut ke **Tahap 9: Modeling** (Logistic
Regression & SVM, minimal 2 algoritma).